In [ ]:
%load_ext ElasticNotebook

In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/small_bench/checkpoints/pre_cell_22.pickle

In [ ]:
%%PandasProfile
### cell 22 ###

import numpy as np

def get_n_grams(n_grams, top_n=10):
    # Prepare texts and types
    texts = train['discourse_text'].astype(str)
    types = train['discourse_type'].values

    # Vectorize into a sparse matrix (use int32 to save memory)
    vec = CountVectorizer(
        lowercase=True,
        stop_words='english',
        ngram_range=(n_grams, n_grams),
        dtype=np.int32
    )
    X = vec.fit_transform(texts)
    feature_names = vec.get_feature_names_out()

    # Sum counts per discourse_type directly on the sparse matrix
    unique_types = sorted(set(types))
    records = []
    for dt in unique_types:
        mask = (types == dt)
        # sum rows for this type (returns 1×n_terms sparse matrix)
        counts = X[mask].sum(axis=0).A1
        # find top_n term indices
        idx = np.argpartition(-counts, top_n - 1)[:top_n]
        idx = idx[np.argsort(-counts[idx])]
        # record (type, term, count)
        for i in idx:
            records.append((dt, feature_names[i], int(counts[i])))

    return pd.DataFrame(records, columns=['Discourse_type', 'words', 'counts'])

# Compute bigrams
bigrams = get_n_grams(n_grams=2, top_n=10)
